# Dwarf Example 25: End-to-End Dwarf Omega Workflow

**EPS Research — Dwarf/Irregular HI Corpus v1.0**

Capstone: full omega workflow on the dwarf corpus.
Compares to SPARC result and prints the full summary.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.20320362  
**Sources:** LVHIS (Koribalski 2019), VLA-ANGST (Ott 2012), LITTLE THINGS (Oh 2015), WALLABY DR2  
**Dependencies:** Python 3, numpy, matplotlib

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'dwarf_irregular_corpus_v1.json': 'https://zenodo.org/records/20320362/files/dwarf_irregular_corpus_v1.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
with open('dwarf_irregular_corpus_v1.json') as f:
    corpus=json.load(f)
results=[]
for g in corpus['galaxies']:
    if not g.get('omega_ready') or not g.get('data') or len(g['data'])<2: continue
    d=g['data']; R=np.array([p['Rad'] for p in d]); V=np.array([p.get('Vrot', 0) for p in d])
    R1,V1=R[0],V[0]; R2,V2=R[-1],V[-1]
    if R1<=0 or R2<=0 or V1<=0 or V2<=0: continue
    omega=(V2/R2-V1/R1)*(R1/R2)**1.5
    V_adj=V-R*omega; V_kep=np.sqrt(V2**2*R2/R)
    gap=(V[-1]-R[-1]*omega)-V[-1]
    results.append({'galaxy':g['galaxy'],'omega':omega,'gap':gap,
                    'vmax':float(max(V)),'survey':g.get('survey','?')})
omegas=[r['omega'] for r in results]
gaps  =[r['gap']   for r in results]
print(f"{'='*55}")
print(f"EPS Research Dwarf Omega Workflow Summary")
print(f"{'='*55}")
print(f"Galaxies analyzed:  {len(results)}")
print(f"Median omega:       {np.median(omegas):.2f} rad/Gyr")
print(f"Mean omega:         {np.mean(omegas):.2f} ± {np.std(omegas):.2f} rad/Gyr")
print(f"All gaps negative:  {all(g<0 for g in gaps)}")
print(f"\nReference values:")
print(f"  SPARC mean omega:   7.06 ± 3.26 rad/Gyr (Flynn & Cannaliato 2025)")
print(f"  Dwarf median omega: 9.94 rad/Gyr (Flynn 2026)")
print(f"  Ratio:              {np.median(omegas)/7.06:.2f}x higher in dwarfs")
fig,ax=plt.subplots(figsize=(7,4))
ax.hist(omegas,bins=12,color='#2ca02c',alpha=0.8,edgecolor='white')
ax.axvline(np.median(omegas),color='red',ls='--',lw=2,label=f'Median={np.median(omegas):.2f}')
ax.axvline(7.06,color='blue',ls=':',lw=2,label='SPARC mean=7.06')
ax.set_xlabel(r'$\omega$ (rad/Gyr)',fontsize=11); ax.set_ylabel('N',fontsize=11)
ax.set_title('Dwarf Omega Distribution — EPS Research\nFlynn (2026)',fontsize=10)
ax.legend(fontsize=9); plt.tight_layout()
plt.savefig('dw25_end_to_end.png',dpi=150,bbox_inches='tight'); plt.show()